# Score Matching — Pre-training Notebook

Trains one `ScoreNet(3→64→64→2, Tanh)` on a 4-cluster 2D Gaussian mixture using
denoising score matching across 10 geometric noise levels.

Output: `src/lessons/score-matching/assets/score-weights.json`

**Run this once before implementing §7 (`<AnnealedLangevin>`).**

In [ ]:
import json, math, numpy as np, torch, torch.nn as nn
np.random.seed(0); torch.manual_seed(0)

## 1. Dataset — 4-cluster 2D GMM

In [ ]:
CENTERS = np.array([[2,2],[2,-2],[-2,2],[-2,-2]], dtype=np.float32)
N_PER   = 250
X = np.vstack([np.random.normal(c, 0.2, (N_PER, 2)) for c in CENTERS]).astype(np.float32)
Xt = torch.tensor(X)
print(f'Dataset: {len(X)} points, 4 clusters at ±2')

## 2. Geometric noise schedule

In [ ]:
SIGMA_MAX, SIGMA_MIN, L = 2.0, 0.01, 10
sigmas = np.exp(np.linspace(np.log(SIGMA_MAX), np.log(SIGMA_MIN), L)).astype(np.float32)
print('Sigma schedule:', [f'{s:.4f}' for s in sigmas])

## 3. Score network: `concat(x, log σ) → score ∈ ℝ²`

In [ ]:
HIDDEN = 64

class ScoreNet(nn.Module):
    def __init__(self, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 2),
        )
    def forward(self, x, log_sigma):
        return self.net(torch.cat([x, log_sigma.unsqueeze(-1)], dim=-1))

model = ScoreNet()
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params}')

## 4. DSM loss with σ² weighting

In [ ]:
def dsm_loss(model, x_clean, sigmas_batch):
    eps = torch.randn_like(x_clean)
    x_noisy = x_clean + sigmas_batch.unsqueeze(-1) * eps
    target = -eps / sigmas_batch.unsqueeze(-1)
    log_s = torch.log(sigmas_batch)
    pred = model(x_noisy, log_s)
    # σ² weighting: all noise levels contribute at comparable scale
    return ((sigmas_batch.unsqueeze(-1) * (pred - target)) ** 2).sum(-1).mean()

## 5. Training

In [ ]:
EPOCHS, BATCH = 15000, 256
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-4)

for epoch in range(EPOCHS):
    idx       = np.random.choice(len(Xt), BATCH, replace=True)
    sig_idx   = np.random.choice(L, BATCH)
    sig_batch = torch.tensor(sigmas[sig_idx])
    loss = dsm_loss(model, Xt[idx], sig_batch)
    opt.zero_grad(); loss.backward(); opt.step(); sch.step()
    if (epoch+1) % 3000 == 0:
        print(f'epoch {epoch+1:5d}  loss={loss.item():.4f}')

print('Training complete.')

## 6. Save weights

In [ ]:
model.eval()

def compact(v):
    if isinstance(v, float): return float(f'{v:.5g}')
    if isinstance(v, list):  return [compact(x) for x in v]
    return v

weights = {k: compact(v.detach().numpy().tolist()) for k, v in model.state_dict().items()}
weights['_metadata'] = {
    'sigmas': sigmas.tolist(),
    'data_centers': CENTERS.tolist(),
    'hidden_dim': HIDDEN,
    'epochs': EPOCHS,
}

OUT = 'src/lessons/score-matching/assets/score-weights.json'
with open(OUT, 'w') as f:
    json.dump(weights, f, separators=(',', ':'))

kb = len(json.dumps(weights, separators=(',',':')).encode()) / 1024
print(f'Saved {OUT} ({kb:.1f} KB)')

## 7. Visual inspection — cosine similarity vs analytical ground truth

Convolution of GMM with N(0, σ²I) yields a GMM with covariances Σ_k + σ²I.

In [ ]:
def logsumexp(a): m=max(a); return m+math.log(sum(math.exp(v-m) for v in a))

def analytical_score(x, sigma):
    var = 0.04 + sigma**2
    log_ws, scores = [], []
    for c in CENTERS:
        d = (x - c).astype(np.float64)
        log_ws.append(-0.5*float(np.sum(d**2))/var - math.log(2*math.pi*var) + math.log(0.25))
        scores.append(-d/var)
    rs = np.exp(np.array(log_ws) - logsumexp(log_ws))
    return sum(r*s for r, s in zip(rs, scores)).astype(np.float32)

def learned_score(x, sigma):
    with torch.no_grad():
        xT  = torch.tensor(x, dtype=torch.float32).unsqueeze(0)
        lsT = torch.tensor([math.log(sigma)], dtype=torch.float32)
        return model(xT, lsT).squeeze(0).numpy()

pts = np.array([[2,2],[2,-2],[-2,2],[-2,-2],[0,0],[1,0],[-1,0],[0,1],[0,-1]], dtype=np.float32)

for name, sigma in [('sigma_max', sigmas[0]), ('sigma_mid', sigmas[L//2]), ('sigma_min', sigmas[-1])]:
    coss = []
    for pt in pts:
        a, l = analytical_score(pt, float(sigma)), learned_score(pt, float(sigma))
        na, nl = np.linalg.norm(a), np.linalg.norm(l)
        if na > 1e-6 and nl > 1e-6:
            coss.append(float(np.dot(a,l)/(na*nl)))
    print(f'{name} (σ={sigma:.4f}): mean cosine={np.mean(coss):.3f}')

## 8. Annealed Langevin cluster recovery test

In [ ]:
np.random.seed(42)
N_PARTICLES, T_INNER, EPSILON = 100, 100, 5e-6
sigma_L = float(sigmas[-1])
parts = np.random.normal(0, float(sigmas[0]), (N_PARTICLES, 2)).astype(np.float32)

for sigma in sigmas:
    sf = float(sigma)
    alpha = EPSILON * (sf / sigma_L) ** 2
    for _ in range(T_INNER):
        sc = np.array([learned_score(p, sf) for p in parts])
        parts = parts + (alpha/2)*sc + math.sqrt(alpha)*np.random.randn(N_PARTICLES, 2).astype(np.float32)

dists = np.array([[np.linalg.norm(p-c) for c in CENTERS] for p in parts])
counts = np.bincount(dists.argmin(1), minlength=4)
w05 = int((dists.min(1) < 0.5).sum())
w10 = int((dists.min(1) < 1.0).sum())
print(f'Cluster counts: {counts.tolist()}')
print(f'Within 0.5: {w05}/100, Within 1.0: {w10}/100')
print('PASS' if w10 >= 70 else 'FAIL — consider retraining with more epochs')